# PALACE — unsupported allocation case

Generate real solver inputs with gsim, measure their sizing requirements, and send those requirements directly to the cloud API before upload and submission. The service owns allocation policy; the category in this filename is an expected test outcome, not an allocation instruction.

Run from this directory with the gsim environment selected as the notebook kernel. Set `SIMULATION_API_URL` to the simulation API base ending in `/v1`, and set `GFP_API_KEY` in the kernel environment. Neither value belongs in saved outputs.

This case attempts cloud preparation and expects an oversized-sizing rejection. It never starts an oversized solve.

Generating this large mesh can take minutes and substantial local RAM. Run the mesh cell on a suitable machine.

In [ ]:
CASE_NAME = "palace-unsupported"
REFINED_MESH_SIZE_UM = 0.2
MAX_MESH_SIZE_UM = 4.0
FIELD_ORDER = 4

In [ ]:
from pathlib import Path

import gdsfactory as gf
from helpers import submit_case, write_request

case_dir = Path.cwd() / "output" / CASE_NAME
input_dir = case_dir / "inputs"
input_dir.mkdir(parents=True, exist_ok=True)

## Configure a coplanar waveguide

Use the public IHP PDK, planar conductors, the selected field order, and a single driven frequency. The mesh-size settings are starting points for the named category, not fixed element counts. Gmsh and PDK versions can change the resulting mesh.

The larger cases use field order 4 to increase Field DOFs without requiring as many tetrahedra. Both measured tetrahedra and estimated Field DOFs are sent to the service.

In [ ]:
from helpers import palace_requirements
from ihp import LAYER, PDK

from gsim.common.stack import get_stack
from gsim.palace import DrivenSim

PDK.activate()
length_um = 800.0
signal_width_um = 20.0
ground_width_um = 40.0
gap_um = 15.0
layer = LAYER.TopMetal2drawing

component = gf.Component()
for y, width in (
    (0.0, signal_width_um),
    ((ground_width_um + signal_width_um) / 2 + gap_um, ground_width_um),
    (-(ground_width_um + signal_width_um) / 2 - gap_um, ground_width_um),
):
    electrode = component << gf.components.rectangle(
        (length_um, width), centered=True, layer=layer
    )
    electrode.move((0, y))
for name, x, orientation in (("o1", -length_um / 2, 180), ("o2", length_um / 2, 0)):
    component.add_port(
        name=name,
        center=(x, 0),
        width=signal_width_um,
        orientation=orientation,
        port_type="electrical",
        layer=layer,
    )

simulation = DrivenSim()
simulation.set_output_dir(input_dir)
simulation.set_geometry(component)
simulation.set_stack(get_stack())
simulation.set_airbox(margin_x=0, margin_y=50, z_above=100, z_below=100)
simulation.solver.order = FIELD_ORDER
simulation.solver.linear.solver_type = "MUMPS"
for port in ("o1", "o2"):
    simulation.add_cpw_port(
        port,
        layer="topmetal2",
        s_width=signal_width_um,
        gap_width=gap_um,
        excited=(port == "o1"),
    )
# A single frequency keeps this an allocation/launch check.
simulation.set_driven(fmin=10e9, fmax=10e9, num_points=1)
validation = simulation.validate_config()
if not validation.valid:
    raise ValueError(str(validation))
print(validation)

## Generate and measure inputs

gsim writes the tetrahedron count and a topology-based Field DOF estimate. The estimate is before Palace preprocessing and can differ from final solver Field DOFs. Inspect the measured JSON and the service’s selected resources; if this case lands in another category, adjust both mesh sizes and rerun. No target count is substituted into the request.

In [ ]:
mesh_result = simulation.mesh(
    preset="default",
    refined_mesh_size=REFINED_MESH_SIZE_UM,
    max_mesh_size=MAX_MESH_SIZE_UM,
    planar_conductors=True,
    auto_size=False,
    fmax=150e9,
    algorithm_3d="delaunay",
    threads=1,
    surface_threads=1,
    verbose=False,
)
config_path = simulation.write_config()
requirements = palace_requirements(input_dir / "metadata.json")
input_files = [mesh_result.mesh_path, config_path]
request = write_request(case_dir, CASE_NAME, requirements)

## Send sizing JSON, upload inputs, and submit

`helpers.submit_case` calls `/job/prepare` with `sizing_requirements`, uploads the input ZIP to the returned presigned URL, then calls `/job/submit`. This explicit notebook adapter avoids older SDK uploads that omit the new field. The target server must support sizing requirements and have sizing profiles enabled.

`metadata.json` and `prepare.json` stay beside `source.zip`; the ZIP contains only the mesh and solver config. This adapter sends metadata directly to the API. Uploading a separate metadata sidecar to S3 still requires SDK/backend support for a sidecar upload URL.

Supported cases print requested CPU and RAM from the server response. Confirm these in the cloud job/admin view against the service’s current profiles. The filename alone does not prove a tier was selected. Failed HTTP requests raise errors; no fallback to fixed-size SDK submission occurs.

In [ ]:
cloud_result = submit_case(
    case_dir,
    request,
    input_files,
    expect_rejection=True,
)
cloud_result